In [17]:
import numpy as np
import joblib
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import load_model

# -----------------------------
# 1. Load tokenizer + models
# -----------------------------
# same tokenizer used for both models
tokenizer = joblib.load("models/tokenizer_bilstm.pkl")

bilstm_model = load_model("models/bilstm_model.h5")
cnn_model    = load_model("models/textcnn_model.h5")

# VERY IMPORTANT: same as used in training
MAX_LEN = 300


# -----------------------------
# 2. Prediction helper
# -----------------------------
def preprocess_text(text: str):
    """Convert raw text to padded sequence."""
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(
        seq,
        maxlen=MAX_LEN,
        padding="post",
        truncating="post"
    )
    return padded


def predict_bilstm(text: str) -> int:
    x = preprocess_text(text)
    probs = bilstm_model.predict(x)
    # trained on labels 0–4, so add 1 to get 1–5
    sentiment = int(np.argmax(probs[0]) + 1)
    return sentiment


def predict_cnn(text: str) -> int:
    x = preprocess_text(text)
    probs = cnn_model.predict(x)
    sentiment = int(np.argmax(probs[0]) + 1)
    return sentiment


# -----------------------------
# 3. Interactive loop
# -----------------------------
print("Choose model:")
print("1. Bi-LSTM")
print("2. Text CNN")

choice = input("Enter 1 or 2: ").strip()

if choice == "1":
    print("\nUsing Bi-LSTM model.")
    while True:
        user_text = input("\nEnter news/article text (or type 'quit'): ")
        if user_text.lower() == "quit":
            break
        score = predict_bilstm(user_text)
        print(f"Bi-LSTM Sentiment Score: {score}  (1 = very negative, 5 = very positive)")

elif choice == "2":
    print("\nUsing Text CNN model.")
    while True:
        user_text = input("\nEnter news/article text (or type 'quit'): ")
        if user_text.lower() == "quit":
            break
        score = predict_cnn(user_text)
        print(f"Text-CNN Sentiment Score: {score}  (1 = very negative, 5 = very positive)")

else:
    print("Invalid choice. Please rerun and enter 1 or 2.")


Choose model:
1. Bi-LSTM
2. Text CNN

Using Text CNN model.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 228ms/step
Text-CNN Sentiment Score: 1  (1 = very negative, 5 = very positive)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
Text-CNN Sentiment Score: 1  (1 = very negative, 5 = very positive)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
Text-CNN Sentiment Score: 4  (1 = very negative, 5 = very positive)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
Text-CNN Sentiment Score: 4  (1 = very negative, 5 = very positive)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
Text-CNN Sentiment Score: 4  (1 = very negative, 5 = very positive)


In [16]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np

# ---------------------------------------------------
# 1. Load tokenizer + trained model
#    (same folder you used in the notebook)
# ---------------------------------------------------
MODEL_DIR = "./distilbert_news_sentiment_final"

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

MAX_LENGTH = 256   # same as in training


# ---------------------------------------------------
# 2. Single-text prediction function
# ---------------------------------------------------
def predict_transformer_sentiment(text: str) -> int:
    """Return sentiment 1–5 for a given news/article text."""
    # Tokenize
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

    # Move to CPU/GPU
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Forward pass (no gradients)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits  # shape (1, 5)

    # Convert logits -> probabilities (optional)
    probs = torch.softmax(logits, dim=-1)

    # Predicted class index 0–4
    class_idx = int(torch.argmax(probs, dim=-1).cpu().item())

    # Your original labels were 1–5 mapped to 0–4, so convert back:
    sentiment_1_to_5 = class_idx + 1
    return sentiment_1_to_5


# ---------------------------------------------------
# 3. Interactive user-input loop
# ---------------------------------------------------
print("DistilBERT News Sentiment (1 = very negative, 5 = very positive)")

while True:
    text = input("\nEnter a news/article text (or type 'quit'): ").strip()
    if text.lower() == "quit":
        break

    score = predict_transformer_sentiment(text)
    print(f"Sentiment Score: {score}  (1 = very negative, 5 = very positive)")


DistilBERT News Sentiment (1 = very negative, 5 = very positive)
Sentiment Score: 1  (1 = very negative, 5 = very positive)
